In [0]:
# Création des menus déroulants/champs texte
dbutils.widgets.text("storage_account", "dataproject2026")
dbutils.widgets.text("container", "raw-data")
dbutils.widgets.text("scope_name", "kv-dataproject2026")
dbutils.widgets.text("secret_key", "storage-key")

# Récupération des valeurs saisies
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
CONTAINER = dbutils.widgets.get("container")
SCOPE = dbutils.widgets.get("scope_name")
SECRET = dbutils.widgets.get("secret_key")

# Configuration de la session Spark
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SCOPE, key=SECRET)
)
print(f" Authentification réussie pour le compte : {STORAGE_ACCOUNT}")

 Authentification réussie pour le compte : dataproject2026


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp

In [0]:
# Schéma basé sur tes données
# name, mpg, cylinders, displacement, horsepower, weight, acceleration, model_year, origin
schema = StructType([
    StructField("name", StringType(), True),
    StructField("mpg", DoubleType(), True),
    StructField("cylinders", IntegerType(), True),
    StructField("displacement", DoubleType(), True),
    StructField("horsepower", DoubleType(), True),
    StructField("weight", IntegerType(), True),
    StructField("acceleration", DoubleType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("origin", StringType(), True)
])

In [0]:
print(schema)

StructType([StructField('name', StringType(), True), StructField('mpg', DoubleType(), True), StructField('cylinders', IntegerType(), True), StructField('displacement', DoubleType(), True), StructField('horsepower', DoubleType(), True), StructField('weight', IntegerType(), True), StructField('acceleration', DoubleType(), True), StructField('model_year', IntegerType(), True), StructField('origin', StringType(), True)])


In [0]:
# Chemins vers ton ADLS Gen2
source_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/input/"
checkpoint_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/checkpoint/"

In [0]:
# LECTURE Streaming
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(schema)
    .load(source_path))

In [0]:
# Ajout d'une colonne de temps pour l'ingestion
df_final = df_stream.withColumn("ingested_at", current_timestamp())

In [0]:
# ÉCRITURE vers la table Delta
query = (df_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime='10 seconds')
    .toTable("cars_streaming_table"))

In [0]:
display(spark.read.table("cars_streaming_table"))

name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,ingested_at
chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,70,usa,2026-04-19T12:24:00.571Z
buick skylark 320,15.0,8,350.0,165.0,3693,11.5,70,usa,2026-04-19T12:24:00.571Z
plymouth satellite,18.0,8,318.0,150.0,3436,11.0,70,usa,2026-04-19T12:24:00.571Z
amc rebel sst,16.0,8,304.0,150.0,3433,12.0,70,usa,2026-04-19T12:24:00.571Z
ford torino,17.0,8,302.0,140.0,3449,10.5,70,usa,2026-04-19T12:24:00.571Z
ford galaxie 500,15.0,8,429.0,198.0,4341,10.0,70,usa,2026-04-19T12:24:00.571Z
chevrolet impala,14.0,8,454.0,220.0,4354,9.0,70,usa,2026-04-19T12:24:00.571Z
plymouth fury iii,14.0,8,440.0,215.0,4312,8.5,70,usa,2026-04-19T12:24:00.571Z
pontiac catalina,14.0,8,455.0,225.0,4425,10.0,70,usa,2026-04-19T12:24:00.571Z
amc ambassador dpl,15.0,8,390.0,190.0,3850,8.5,70,usa,2026-04-19T12:24:00.571Z


In [0]:
%sql
DESCRIBE HISTORY cars_streaming_table;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-04-19T12:26:42Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> d208d18b-0c57-4ee7-86ab-4d779cfe6f45, epochId -> 2, statsOnLoad -> true)",null,List(3371021982898756),0418-133059-gw75dl87,6,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 19, numOutputBytes -> 3398, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
6,2026-04-19T12:25:51Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> d208d18b-0c57-4ee7-86ab-4d779cfe6f45, epochId -> 1, statsOnLoad -> true)",null,List(3371021982898756),0418-133059-gw75dl87,5,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 17, numOutputBytes -> 3410, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
5,2026-04-19T12:24:05Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> d208d18b-0c57-4ee7-86ab-4d779cfe6f45, epochId -> 0, statsOnLoad -> true)",null,List(3371021982898756),0418-133059-gw75dl87,4,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 15, numOutputBytes -> 3285, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
4,2026-04-19T11:15:17Z,145105645084921,youssefjabrouun@gmail.com,DELETE,"Map(predicate -> [""true""])",null,List(3371021982898756),0418-133059-gw75dl87,3,WriteSerializable,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 10093, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 883, numDeletionVectorsUpdated -> 0, numDeletedRows -> 51, scanTimeMs -> 813, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/17.3.x-photon-scala2.13
3,2026-04-18T14:15:22Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 9de8b12b-a512-4915-8048-3be97e92d224, epochId -> 2, statsOnLoad -> false)",null,List(3371021982898756),0418-133059-gw75dl87,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 19, numOutputBytes -> 3398, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
2,2026-04-18T14:12:02Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 9de8b12b-a512-4915-8048-3be97e92d224, epochId -> 1, statsOnLoad -> false)",null,List(3371021982898756),0418-133059-gw75dl87,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 17, numOutputBytes -> 3410, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
1,2026-04-18T13:59:57Z,145105645084921,youssefjabrouun@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 9de8b12b-a512-4915-8048-3be97e92d224, epochId -> 0, statsOnLoad -> true)",null,List(3371021982898756),0418-133059-gw75dl87,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 15, numOutputBytes -> 3285, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-photon-scala2.13
0,2026-04-18T13:59:38Z,145105645084921,youssefjabrouun@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-0d8199d7-04f4-45c1-8a6d-81158a4d5416"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-f843f545-53cb-48b9-8daa-62c8960f6882""}, statsOnLoad -> false)",null,List(3371021982898756),0418-133059-gw75dl87,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-photon-scala2.13
